# Site checklist table

Enter a Massachusetts latitude and longitude to get a Polars table with exactly four columns:

`checklist item`, `subcategory_name`, `subcategory_value`, `data_source`.

Run from the repository environment after `uv pip install -r requirements.txt`. Launch with `uv run --env-file .env jupyter lab site_checklist.ipynb`. This notebook calls the Python backend directly.

Missing values stay null. Values are text because this column combines addresses, numbers and other data. Units remain in field names such as `ground_elevation_m` and `annual_electricity_kwh`. Candidate field names stay labeled, and numbered candidates keep their attributes together. Municipal solar context is separate from the facility's actual solar status.

In [1]:
import polars as pl
from IPython.display import display
from API import lookup_table

LATITUDE = 42.359
LONGITUDE = -71.093

table = lookup_table(LATITUDE, LONGITUDE)
with pl.Config(tbl_rows=-1, fmt_str_lengths=500):
    display(table)

checklist item,subcategory_name,subcategory_value,data_source
str,str,str,str
"""Site identity and boundaries""","""address""","""60 VASSAR ST""","""https://services1.arcgis.com/hGdibHYSPO59RG1h/arcgis/rest/services/Massachusetts_Property_Tax_Parcels/FeatureServer/0"""
"""Site identity and boundaries""","""municipality""","""CAMBRIDGE""","""https://arcgisserver.digital.mass.gov/arcgisserver/rest/services/AGOL/ElectricityProviders/MapServer/0"""
"""Site identity and boundaries""","""state""","""MA""","""https://arcgisserver.digital.mass.gov/arcgisserver/rest/services/AGOL/ElectricityProviders/MapServer/0"""
"""Site identity and boundaries""","""latitude""","""42.359""","""User-provided coordinates"""
"""Site identity and boundaries""","""longitude""","""-71.093""","""User-provided coordinates"""
"""Site identity and boundaries""","""parcel_id""","""F_766725_2956445""","""https://services1.arcgis.com/hGdibHYSPO59RG1h/arcgis/rest/services/Massachusetts_Property_Tax_Parcels/FeatureServer/0"""
"""Site identity and boundaries""","""building_use""","""College or University (Educational Private)""","""https://services1.arcgis.com/hGdibHYSPO59RG1h/arcgis/rest/services/Massachusetts_Property_Tax_Parcels/FeatureServer/0"""
"""Site identity and boundaries""","""assessor_building_area_sqft""","""56880""","""https://services1.arcgis.com/hGdibHYSPO59RG1h/arcgis/rest/services/Massachusetts_Property_Tax_Parcels/FeatureServer/0"""
"""Contacts and authority""","""recorded_owner""","""MASSACHUSETTS INSTITUTE OF TECHNOLOGY""","""https://services1.arcgis.com/hGdibHYSPO59RG1h/arcgis/rest/services/Massachusetts_Property_Tax_Parcels/FeatureServer/0"""


## Save the table

The CSV is convenient to share or open in Excel. Parquet preserves nulls. Both files are saved under Git-ignored `data/processed/`.

In [2]:
from pathlib import Path

output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
table.write_csv(output_dir / "site_checklist.csv")
table.write_parquet(output_dir / "site_checklist.parquet")
print(f"Saved {table.height} rows with columns {table.columns}")

Saved 92 rows with columns ['checklist item', 'subcategory_name', 'subcategory_value', 'data_source']


## Source errors

Errors appear as separate `*.error` rows, while missing checklist values stay null. No automatic retry or fallback hides a failed retrieval.

OpenEI HTTP 429 means its rate limit was exceeded. Set a personal `OPENEI_API_KEY` in `.env`. Restart Jupyter with `uv run --env-file .env jupyter lab site_checklist.ipynb` after editing the file, so new kernels inherit the key. For the HTTP API, restart the API process so it inherits the key.

In [3]:
errors = table.filter(pl.col("subcategory_name").str.ends_with(".error"))
if errors.height:
    with pl.Config(tbl_rows=-1, fmt_str_lengths=1000):
        display(errors)
else:
    print("No provider errors returned for this lookup.")

No provider errors returned for this lookup.


The same four-column result is available as JSON rows over HTTP:

```python
import requests

response = requests.get(
    "http://127.0.0.1:8001/site/table",
    params={"latitude": LATITUDE, "longitude": LONGITUDE},
    timeout=120,
)
response.raise_for_status()
table = pl.from_dicts(response.json())
```

If you already have the full `/site` response, use `from API import checklist_table` followed by `table = checklist_table(site)`. This formats the existing response without fetching providers again. Full provenance, units, review statuses and original records remain available in the full response. Archived blueprints and customer documents still require the follow-up actions listed there.